# Agentic-RAG Evidence 提取调试 Notebook逐步运行每个 cell，查看 LLM 返回的原始 JSON 和解析结果。出错直接在当前 cell 看到完整 traceback。

In [ ]:
import sys, json, re, tracebackfrom pathlib import Pathsys.path.insert(0, str(Path.cwd()))print("Ready.")

## 1. 选择测试文件（修改下面路径）

In [ ]:
FILE_PATH = "/home/wrz/code/agentic-rag/data/raw/中华人民共和国劳动法_20181229.docx"print(f"File: {FILE_PATH}")

## 2. 解析文档 (MinerU)

In [ ]:
from src.document_parser.mineru_parser import MinerUParserdoc_id = Path(FILE_PATH).stemparser = MinerUParser()result = parser.parse(FILE_PATH, doc_id)markdown = result.get("markdown", "")print(f"Markdown: {len(markdown)} chars")print(markdown[:300])

## 3. Chunk 切分

In [ ]:
from config.settings import settingsfrom src.chunker.hierarchical_chunker import HierarchicalChunkerfrom src.chunker.table_chunker import TableChunkerfrom src.chunker.context_merger import ContextMergerfrom src.document_parser.table_parser import TableParserh_chunker = HierarchicalChunker(chunk_size=settings.chunk_size, chunk_overlap=settings.chunk_overlap)chunks = h_chunker.chunk(doc_id, markdown)tables = TableParser.extract_tables_from_markdown(markdown)chunks.extend(TableChunker().chunk_tables(doc_id, tables))chunks = ContextMerger().merge(chunks)chunks = ContextMerger().deduplicate(chunks)chunks = [c for c in chunks if c["text"].strip()]print(f"Chunks: {len(chunks)}")for i, c in enumerate(chunks):    print(f"  [{i}] {len(c['text'])} chars | {c['text'][:100]}...")

## 4. 选中一个 chunk 进行测试（默认第0个）

In [ ]:
CHUNK_IDX = 0  # 修改这里选择不同 chunktext = chunks[CHUNK_IDX]["text"][:4000]print(f"=== Chunk [{CHUNK_IDX}] ({len(text)} chars) ===")print(text)

## 5. 🚀 调用 LLM（会消耗 token！）

In [ ]:
from src.knowledge_graph.entity_extractor import EntityExtractorfrom config.prompts import EVIDENCE_EXTRACTION_SYSTEM, EVIDENCE_EXTRACTION_EXAMPLEextractor = EntityExtractor()llm = extractor._get_llm()prompt = f"{EVIDENCE_EXTRACTION_SYSTEM}Example:{EVIDENCE_EXTRACTION_EXAMPLE}Text: {text}Response:"resp = llm.invoke(prompt)raw = resp.contentprint(f"=== Raw LLM response ({len(raw)} chars) ===")print(raw)

## 6. 逐步解析：regex → json.loads → _parse_evidence_items

In [ ]:
print("=== Step A: regex extract JSON ===")try:    match = re.search(r"\{[\s\S]*\}", raw)    json_str = match.group() if match else "NO MATCH"    print(f"Extracted {len(json_str)} chars")except Exception as e:    print(f"REGEX ERROR: {e}"); traceback.print_exc()    json_str = "{}"print()print("=== Step B: json.loads ===")try:    data = json.loads(json_str)    print(f"Type: {type(data).__name__}")    if isinstance(data, dict):        print(f"Keys: {list(data.keys())}")        ev = data.get("evidence_items", "MISSING")        print(f"evidence_items type: {type(ev).__name__}")        if isinstance(ev, list):            for i, it in enumerate(ev[:5]):                print(f"  [{i}] type={type(it).__name__}: {str(it)[:200]}")        elif isinstance(ev, str):            print(f"STRING: {ev[:500]}")    else:        print(f"NOT A DICT: {str(data)[:500]}")except Exception as e:    print(f"JSON ERROR: {e}"); traceback.print_exc()    data = {}print()print("=== Step C: _parse_evidence_items (defensive) ===")from src.knowledge_graph.entity_extractor import _parse_evidence_itemstry:    items = _parse_evidence_items(data)    print(f"Parsed {len(items)} items")    for i, it in enumerate(items[:5]):        print(f"  [{i}] type={it.get('type','?')} name={it.get('entity_name', it.get('subject_name','?'))}")except Exception as e:    print(f"PARSE ERROR: {e}"); traceback.print_exc()    items = []

## 7. 构造 Evidence 对象（这里最容易出 string indices 错误）

In [ ]:
from src.knowledge_graph.evidence import (    EntityEvidence, EntityAttributeEvidence, FactEvidence, FactAttributeEvidence,)chunk_hash = "debug_chunk_test"objs = []for idx, item in enumerate(items):    print(f"  [{idx}] item type: {type(item).__name__}", end=" ")    try:        etype = item.get("type", "").upper()        conf = float(item.get("confidence", 0.5))        ev_text = text[:200]        print(f"evidence_type={etype}", end=" ")        if etype == "ENTITY":            ev = EntityEvidence(chunk_hash, item.get("entity_name",""), item.get("entity_type","?"), conf, ev_text)        elif etype == "ENTITY_ATTRIBUTE":            ev = EntityAttributeEvidence(chunk_hash, item.get("entity_key",""), item.get("attr_key",""), item.get("attr_value",""), conf, ev_text)        elif etype == "FACT":            ev = FactEvidence(chunk_hash, item.get("subject_name",""), item.get("subject_type","?"), item.get("predicate",""), item.get("object_name",""), item.get("object_type","?"), conf, ev_text)        elif etype == "FACT_ATTRIBUTE":            ev = FactAttributeEvidence(chunk_hash, item.get("subject_key",""), item.get("predicate",""), item.get("object_key",""), item.get("attr_key",""), item.get("attr_value",""), conf, ev_text)        else:            print(f"→ SKIP (unknown)")            continue        objs.append(ev)        print(f"→ {ev.affected_key}")    except Exception as e:        print(f"→ ERROR: {e}")        print(f"    item = {item}")        traceback.print_exc()print(f"Built {len(objs)} evidence objects")

## 8. 写入 SQLite 并验证

In [ ]:
from src.knowledge_graph.evidence_writer import write_evidencefrom src.storage import doc_storeprint("Writing to SQLite...")result = write_evidence(chunk_hash, objs)print(f"Wrote {result['count']} records")conn = doc_store._get_conn()rows = conn.execute(    "SELECT evidence_type, COUNT(*) as cnt FROM evidence WHERE chunk_hash=? AND active=1 GROUP BY 1",    (chunk_hash,),).fetchall()for r in rows:    print(f"  {r['evidence_type']}: {r['cnt']}")conn.close()print("Done.")